# Notebook 8: RaGNAR-style random-network search

Controlled low-order random-graph search using a fixed validation/evaluation split and a common conjugate prior across all candidate graphs and internal baselines. Absolute CRPS values are interpreted only within this experiment.

In [ ]:
import importlib, shared_utils
importlib.reload(shared_utils)
from shared_utils import *
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Search configuration.
P = 2
STAGES = [1, 1]
QUICK = quick_mode()                    # True for a fast execution check
PI_LADDER = [0.10, 0.15, 0.20]   # Erdos-Renyi edge probabilities (~2-4 stage-1 neighbours)
N_GRAPHS = 3000                  # random graphs per replication
TOP_N = 5                        # average the best TOP_N graphs
N_REPS = 10                      # replications for error bars
N_VAL = 30                       # validation block (ranking), months
N_EVAL = None                    # evaluation block; None uses all remaining test months
PRIOR = dict(prior_type="minnesota", local_alpha=False,
             lambda1=0.5, lambda2=0.5, lambda3=1.0, rw_centre=1.0)

if QUICK:
    N_GRAPHS, N_REPS = 20, 2

# The random graphs replace the observed topology in the experiment. The shared loader
# still reads the standard project inputs, but the observed matrices are not used below.
d = load_data(network="geographic")
Y_full = d["Y_full"].to_numpy()
test_start = d["test_start"]
N = d["N"]; countries = d["countries"]
n_test_total = Y_full.shape[0] - test_start
if N_EVAL is None:
    N_EVAL = n_test_total - N_VAL
print(f"N={N} countries, {Y_full.shape[0]} months")
print(f"test months={n_test_total}: validation={N_VAL}, evaluation={N_EVAL}")
assert N_EVAL > 5, "eval block too small"

## Random-graph sanity check

Confirm the chosen edge probabilities give small stage-1 neighbourhoods (RaGNAR targets
around three) and non-empty stage-2 sets, so the second-stage regressor is identified,
whereas complete adjacency has no exact stage-2 neighbours.

In [ ]:
rng = np.random.default_rng(0)
print(f"{'pi':>6} {'mean stage-1 nbrs':>18} {'isolated/graph':>15} {'stage-2 edges':>14}")
for pi in PI_LADDER:
    degs, iso, s2 = [], 0, []
    for _ in range(300):
        A = erdos_renyi_graph(N, pi, rng)
        dd = A.sum(1); degs.append(dd.mean()); iso += int((dd == 0).sum())
        sw = ragnar_stage_weights(A); s2.append(int((sw[1] > 0).sum()))
    print(f"{pi:>6.2f} {np.mean(degs):>18.5f} {iso/300:>15.5f} {np.mean(s2):>14.5g}")

## Run the search across the edge-probability ladder

For each edge probability: rank the random graphs by validation CRPS, average the top
few, evaluate on the held-out block, and repeat across replications. The two baselines
are fitted once per level on the same evaluation block. This is the compute-heavy cell.

In [ ]:
t0 = time.time()
results = {}
for pi in PI_LADDER:
    print(f"\npi = {pi}")
    res = ragnar_search(
        Y_full, test_start=test_start, n_val=N_VAL, n_eval=N_EVAL,
        p=P, s=STAGES, prior_kwargs=PRIOR,
        n_graphs=N_GRAPHS, pi=pi, top_n=TOP_N, n_reps=N_REPS,
        BayesianGNARClass=BayesianGNAR, base_seed=0, verbose=False)
    results[pi] = res
    print(f"  AR-only CRPS     = {res['ar_crps']:.5f}")
    print(f"  AR+factor CRPS   = {res['factor_crps']:.5f}")
    print(f"  top-{TOP_N} CRPS      = {res['top_crps_mean']:.5f} +/- {res['top_crps_std']:.5f}")
    print(f"  improvement vs AR     = {res['improvement_vs_ar']:+.5f}")
    print(f"  improvement vs factor = {res['improvement_vs_factor']:+.5f}")
print(f"\ntotal search time: {(time.time()-t0)/60:.5g} min")

## Random-network comparison

Evaluation CRPS for the selected random-network ensembles is compared with the matched AR-only and uniform cross-sectional baselines. Error bars show one standard deviation across random-search replications.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(PI_LADDER))
top_means = [results[p]["top_crps_mean"] for p in PI_LADDER]
top_stds = [results[p]["top_crps_std"] for p in PI_LADDER]
ar = [results[p]["ar_crps"] for p in PI_LADDER]
fac = [results[p]["factor_crps"] for p in PI_LADDER]

ax.errorbar(x, top_means, yerr=top_stds, marker="o", capsize=4,
            label=f"top-{TOP_N} random networks", lw=2)
ax.plot(x, ar, marker="s", ls="--", label="AR-only baseline")
ax.plot(x, fac, marker="^", ls="--", label="AR + common-factor baseline")
ax.set_xticks(x); ax.set_xticklabels([f"pi={p}" for p in PI_LADDER])
ax.set_ylabel("evaluation CRPS (lower is better)")
ax.set_title("RaGNAR search vs baselines")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Search variability across random-graph replications

The ten replications quantify sensitivity to the random candidate-graph draw. They are
not independent forecast samples, so no inferential test is attached to their dispersion.
The relevant descriptive question is whether the selected random-network ensembles
consistently approach or improve on the same-window AR-only and uniform-factor
baselines.

In [ ]:
rows = []
for pi in PI_LADDER:
    r = results[pi]
    rep_scores = np.asarray(r["top_crps_all"], dtype=float)
    rows.append({
        "pi": pi,
        "mean_top_crps": float(rep_scores.mean()),
        "sd_top_crps": float(rep_scores.std(ddof=1)),
        "min_top_crps": float(rep_scores.min()),
        "max_top_crps": float(rep_scores.max()),
        "ar_crps": float(r["ar_crps"]),
        "factor_crps": float(r["factor_crps"]),
        "mean_gain_vs_ar": float(r["improvement_vs_ar"]),
        "mean_gain_vs_factor": float(r["improvement_vs_factor"]),
    })

search_variability = pd.DataFrame(rows)
print(search_variability.round(5).to_string(index=False))

## Reading the result

The random-network search is interpreted only against its own two baselines on the same
72-month evaluation block. If the selected ensembles improve on AR-only but not on the
uniform-factor model, the experiment provides no evidence that the searched random
topology contributes predictive information beyond broad cross-sectional averaging.
Variation across the ten replications describes sensitivity to the random candidate set,
not sampling uncertainty in the forecast-loss process.

In [ ]:
save_result("nb8_ragnar_search", {
    "search_config": dict(p=P, s=STAGES, pi_ladder=PI_LADDER, n_graphs=N_GRAPHS,
                          top_n=TOP_N, n_reps=N_REPS, n_val=N_VAL, n_eval=N_EVAL),
    "results": {str(pi): {k: v for k, v in results[pi].items() if k != "reps"}
                for pi in PI_LADDER},
    "search_variability": search_variability.to_dict(orient="records"),
    "window_note": "the search splits the standard test window into a validation block "
                   "used to rank graphs and a disjoint evaluation block, so absolute "
                   "CRPS here is not comparable with any other notebook",
    "config": run_config(p=P, stages=STAGES, purpose="low_order_topology_search"),
    "quick": QUICK,
})
print("saved nb8_ragnar_search")